# AnyText test on Colab GPU

Untested starter notebook. Run the cells in order, one at a time (Shift+Enter).

Before you start: connect this notebook to your Colab server (kernel = Colab, GPU), and upload the `AnyText` folder with right-click > Upload to Colab.

## 1. Check the GPU and the uploaded folder

In [ ]:
!nvidia-smi
!ls /content

In [ ]:
%cd /content/AnyText
!ls
!ls font

## 2. Install and smoke test

The first run downloads the pretrained weights, which can take several minutes.
If it stops with `No module named X`, run `!pip install X` and run this cell again.

In [ ]:
!pip -q install modelscope
!python inference.py

## 3. Test on ONE statement

Edit `IMG`, `BOX` and `NEW_TEXT` first.
- `IMG`: path of the statement image on the Colab server (check with `!ls /content`).
- `BOX`: region to edit in pixels on the full page: [x0, y0, x1, y1].
- `NEW_TEXT`: the text to write there.

Compare `mask.png` with `example_images/edit7.png` to check the mask format.

In [ ]:
import glob
from PIL import Image, ImageDraw
from IPython.display import display
from modelscope.pipelines import pipeline
from util import save_images

pipe = pipeline('my-anytext-task', model='damo/cv_anytext_text_generation_editing', model_revision='v1.1.3')

IMG = "PATH_TO_PAGE"
BOX = [100, 200, 300, 240]
NEW_TEXT = "1,250.00"
CROP = 512

img = Image.open(IMG).convert("RGB")
W, H = img.size
s = min(CROP, W, H)
cx, cy = (BOX[0] + BOX[2]) // 2, (BOX[1] + BOX[3]) // 2
left = min(max(cx - s // 2, 0), W - s)
top = min(max(cy - s // 2, 0), H - s)
crop = img.crop((left, top, left + s, top + s))
mask = Image.new("L", crop.size, 0)
ImageDraw.Draw(mask).rectangle([BOX[0] - left, BOX[1] - top, BOX[2] - left, BOX[3] - top], fill=255)
crop.save("crop.png")
mask.save("mask.png")

input_data = {
    "prompt": f'A bank statement with the text "{NEW_TEXT}"',
    "seed": 1234,
    "draw_pos": "mask.png",
    "ori_image": "crop.png",
}
results, rtn_code, rtn_warning, debug_info = pipe(
    input_data, mode="text-editing", show_debug=True, image_count=2, ddim_steps=20)
if rtn_code >= 0:
    save_images(results, "SaveImages_test")
if rtn_warning:
    print(rtn_warning)

display(crop, mask)
for p in sorted(glob.glob("SaveImages_test/*")):
    display(Image.open(p))